# Chapter 12: Structure Computation

Source orientation: printed pages 310-324; PDF pages 328-342. This notebook uses the source span for section names and mathematical coverage only; the prose, data, code, and figures here are original.

Chapter 12 asks a practical two-view question: once the cameras are known, where is the 3D point that produced a noisy image match? The subtlety is that the measured back-projected rays usually miss each other. A triangulation routine is therefore an estimator, and the right question is not just "which point do the rays define?" but "which image-consistent point minimizes the geometry we can actually trust?

## Chapter Goal

By the end of this notebook you should be able to inspect structure computation as a chain of testable geometric choices:

- build the linear equations used by homogeneous and inhomogeneous triangulation;
- compare midpoint, linear, Sampson-corrected, and optimal image-corrected triangulation;
- check cheirality and depth signs, not only reprojection residuals;
- see why uncertainty stretches along viewing rays when the ray angle is small;
- reconstruct a 3D line as the intersection of two back-projected image planes and recognize the epipolar-plane degeneracy.

## Translation Guide

| Book concept | Computational representation in this notebook | Inspection target |
| --- | --- | --- |
| Back-projected ray | camera center plus direction `inv(P[:, :3]) @ x` | noisy corresponding rays are skew, so the closest segment has nonzero length |
| Linear triangulation | SVD null vector of the four equations `A X = 0` | residuals are small, but the method minimizes an algebraic quantity |
| Inhomogeneous triangulation | least squares with `X_4 = 1` | stable for finite Euclidean points, fragile near points at infinity |
| Midpoint triangulation | midpoint of the common perpendicular to two Euclidean rays | useful in calibrated Euclidean space, not projective-invariant |
| Sampson correction | first-order correction to the epipolar constraint | cheap image correction that nearly enforces `x2.T @ F @ x1 = 0` |
| Optimal image correction | OpenCV's `correctMatches`, the Hartley-Sturm style two-view correction | corrected points satisfy the epipolar constraint to numerical precision |
| Structure refinement | nonlinear least squares in 3D point coordinates | reprojection error decreases from the linear initialization |
| Reconstruction uncertainty | inverse normal matrix of the projection Jacobian | largest variance grows as the two viewing rays become nearly parallel |
| Line reconstruction | two planes `P.T @ l` and `P'.T @ l'` intersect in a 3D line | lines in an epipolar plane are weakly determined from two views |

## Route Through The Chapter

The notebook follows the chapter's computation order: first the noisy-ray problem, then linear triangulation, then image-space geometric error, then uncertainty, then line reconstruction. The lab uses only synthetic measurements generated from known cameras, so every visual has a numeric invariant: epipolar residual, reprojection error, depth sign, covariance eigenvalue, or plane-line incidence.

In [ ]:
from pathlib import Path
import sys

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate the Multiple View Geometry book root.")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

TOPIC = "chapter-12"
artifact_paths = []
check_paths = []
BOOK_ROOT

In [ ]:
import json
import math

import cv2
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
from scipy import linalg, optimize

from utils.artifacts import (
    assert_artifacts,
    display_artifact,
    relative_to_book,
    save_csv,
    save_json,
    save_matplotlib,
    save_plotly_html,
)
from utils.cameras import (
    camera_center,
    camera_matrix,
    look_at_rotation,
    make_calibration,
    project_points,
    synthetic_cameras,
)
from utils.epipolar import fundamental_from_cameras, linear_triangulate, sampson_errors
from utils.projective import dehomogenize, homogenize

rng = np.random.default_rng(12012)
np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

## Library Routing

This is a projective computer-vision chapter. NumPy and SciPy expose the linear algebra, SVD nullspaces, Jacobians, and nonlinear least-squares refinement. OpenCV is used where the chapter calls for the standard optimal two-view correction behind `correctMatches`, rather than reimplementing the sixth-degree polynomial. Plotly is used for 3D ray, camera, uncertainty, and plane-intersection inspection because rotation matters. Matplotlib is used for durable 2D image-plane overlays and scalar cost curves. The final checks assert geometry, not just file creation.

In [ ]:
def make_structure_scene(n=36, noise_sigma=1.5):
    # Known two-camera scene with noisy image correspondences.
    K, P1, P2 = synthetic_cameras()
    points3d = np.column_stack([
        rng.uniform(-0.9, 0.9, n),
        rng.uniform(-0.6, 0.7, n),
        rng.uniform(2.4, 4.6, n),
    ])
    x1_clean = project_points(P1, points3d)
    x2_clean = project_points(P2, points3d)
    x1_noisy = x1_clean + rng.normal(scale=noise_sigma, size=x1_clean.shape)
    x2_noisy = x2_clean + rng.normal(scale=noise_sigma, size=x2_clean.shape)
    return K, P1, P2, points3d, x1_clean, x2_clean, x1_noisy, x2_noisy


def camera_center_euclidean(P):
    return dehomogenize(camera_center(P).reshape(1, 4))[0]


def ray_from_image(P, x):
    C = camera_center_euclidean(P)
    direction = np.linalg.inv(P[:, :3]) @ np.array([x[0], x[1], 1.0])
    direction = direction / np.linalg.norm(direction)
    return C, direction


def closest_points_on_rays(C1, d1, C2, d2):
    A = np.column_stack([d1, -d2])
    st, *_ = np.linalg.lstsq(A, C2 - C1, rcond=None)
    q1 = C1 + st[0] * d1
    q2 = C2 + st[1] * d2
    return q1, q2, 0.5 * (q1 + q2), np.linalg.norm(q1 - q2)


def dlt_matrix(P1, P2, x1, x2):
    u1, v1 = x1[:2]
    u2, v2 = x2[:2]
    return np.vstack([
        u1 * P1[2] - P1[0],
        v1 * P1[2] - P1[1],
        u2 * P2[2] - P2[0],
        v2 * P2[2] - P2[1],
    ])


def inhomogeneous_triangulate(P1, P2, points1, points2):
    out = []
    for a, b in zip(points1, points2):
        A = dlt_matrix(P1, P2, a, b)
        xyz, *_ = np.linalg.lstsq(A[:, :3], -A[:, 3], rcond=None)
        out.append(xyz)
    return np.asarray(out)


def midpoint_triangulate(P1, P2, points1, points2):
    out = []
    gaps = []
    for a, b in zip(points1, points2):
        C1, d1 = ray_from_image(P1, a)
        C2, d2 = ray_from_image(P2, b)
        _, _, mid, gap = closest_points_on_rays(C1, d1, C2, d2)
        out.append(mid)
        gaps.append(gap)
    return np.asarray(out), np.asarray(gaps)


def sampson_correct(F, points1, points2):
    x1h = homogenize(points1)
    x2h = homogenize(points2)
    Fx1 = (F @ x1h.T).T
    Ftx2 = (F.T @ x2h.T).T
    err = np.sum(x2h * Fx1, axis=1)
    denom = Fx1[:, 0] ** 2 + Fx1[:, 1] ** 2 + Ftx2[:, 0] ** 2 + Ftx2[:, 1] ** 2
    scale = -err / np.maximum(denom, 1e-12)
    delta1 = np.column_stack([scale * Ftx2[:, 0], scale * Ftx2[:, 1]])
    delta2 = np.column_stack([scale * Fx1[:, 0], scale * Fx1[:, 1]])
    return points1 + delta1, points2 + delta2


def optimal_correct(F, points1, points2):
    c1, c2 = cv2.correctMatches(
        F.astype(np.float64),
        points1.reshape(1, -1, 2).astype(np.float64),
        points2.reshape(1, -1, 2).astype(np.float64),
    )
    return c1.reshape(-1, 2), c2.reshape(-1, 2)


def epipolar_residuals(F, points1, points2):
    x1h = homogenize(points1)
    x2h = homogenize(points2)
    return np.sum(x2h * (F @ x1h.T).T, axis=1)


def projection_residual_vector(P1, P2, X, x1, x2):
    X = np.asarray(X, dtype=float)
    return np.r_[project_points(P1, X.reshape(1, 3))[0] - x1, project_points(P2, X.reshape(1, 3))[0] - x2]


def refine_structure_points(P1, P2, initial_points, points1, points2):
    refined = []
    for X0, a, b in zip(initial_points, points1, points2):
        result = optimize.least_squares(
            lambda Y: projection_residual_vector(P1, P2, Y, a, b),
            X0,
            method="trf",
            max_nfev=80,
            xtol=1e-11,
            ftol=1e-11,
            gtol=1e-11,
        )
        refined.append(result.x)
    return np.asarray(refined)


def reprojection_rmse(P1, P2, points3d, points1, points2):
    r1 = project_points(P1, points3d) - points1
    r2 = project_points(P2, points3d) - points2
    return float(np.sqrt(np.mean(np.sum(r1 * r1, axis=1) + np.sum(r2 * r2, axis=1))))


def mean_world_error(points3d, truth):
    return float(np.mean(np.linalg.norm(points3d - truth, axis=1)))


def camera_depths(P, points3d):
    return (P[2] @ homogenize(points3d).T).T


def ray_angle_degrees(P1, P2, X):
    C1 = camera_center_euclidean(P1)
    C2 = camera_center_euclidean(P2)
    d1 = X - C1
    d2 = X - C2
    d1 = d1 / np.linalg.norm(d1)
    d2 = d2 / np.linalg.norm(d2)
    return float(np.degrees(np.arccos(np.clip(np.dot(d1, d2), -1.0, 1.0))))


def numerical_projection_jacobian(P1, P2, X, eps=1e-5):
    J = np.zeros((4, 3))
    for j in range(3):
        step = np.zeros(3)
        step[j] = eps
        plus = np.r_[project_points(P1, (X + step).reshape(1, 3))[0], project_points(P2, (X + step).reshape(1, 3))[0]]
        minus = np.r_[project_points(P1, (X - step).reshape(1, 3))[0], project_points(P2, (X - step).reshape(1, 3))[0]]
        J[:, j] = (plus - minus) / (2.0 * eps)
    return J


def point_covariance(P1, P2, X, sigma_pixels=1.0):
    J = numerical_projection_jacobian(P1, P2, X)
    info = J.T @ J
    cov = sigma_pixels**2 * np.linalg.pinv(info)
    return 0.5 * (cov + cov.T)


def add_plotly_segment(fig, a, b, name, color, width=5, dash=None):
    line = dict(color=color, width=width)
    if dash is not None:
        line["dash"] = dash
    fig.add_trace(go.Scatter3d(
        x=[a[0], b[0]], y=[a[1], b[1]], z=[a[2], b[2]],
        mode="lines", name=name, line=line,
    ))


def epiline_segment(line, xlim, ylim):
    a, b, c = line
    candidates = []
    for x in xlim:
        if abs(b) > 1e-12:
            y = -(a * x + c) / b
            if ylim[0] <= y <= ylim[1]:
                candidates.append((x, y))
    for y in ylim:
        if abs(a) > 1e-12:
            x = -(b * y + c) / a
            if xlim[0] <= x <= xlim[1]:
                candidates.append((x, y))
    if len(candidates) < 2:
        return None
    return np.asarray(candidates[:2])


def geometric_cost(theta, f, fp, a, b, c, d):
    t = np.tan(theta)
    return t**2 / (1.0 + (f * t) ** 2) + (c * t + d) ** 2 / ((a * t + b) ** 2 + (fp * (c * t + d)) ** 2)


def polynomial_roots_for_cost(f, fp, a, b, c, d):
    t = np.poly1d([1.0, 0.0])
    p = np.poly1d([a, b])
    q = np.poly1d([c, d])
    denom2 = p * p + (fp**2) * q * q
    one_plus = np.poly1d([f**2, 0.0, 1.0])
    g = t * (denom2**2) - (a * d - b * c) * (one_plus**2) * p * q
    return np.roots(g.c), g


def line_from_projected_endpoints(P, A, B):
    xA = homogenize(project_points(P, A.reshape(1, 3)))[0]
    xB = homogenize(project_points(P, B.reshape(1, 3)))[0]
    line = np.cross(xA, xB)
    return line / np.linalg.norm(line[:2])


def nullspace_line_points(plane_stack):
    _, _, vt = linalg.svd(plane_stack)
    return vt[-2], vt[-1]


def affine_points_from_line_basis(U, V, count=2):
    vals = []
    for s in np.linspace(-1.0, 1.0, count):
        X = U + s * V
        if abs(X[-1]) > 1e-9:
            vals.append(X[:3] / X[-1])
    return np.asarray(vals)


def plane_surface(plane, span=3.0, resolution=8):
    n = np.asarray(plane[:3], dtype=float)
    d = float(plane[3])
    origin = -d * n / np.dot(n, n)
    helper = np.array([1.0, 0.0, 0.0]) if abs(n[0]) < 0.8 else np.array([0.0, 1.0, 0.0])
    u = np.cross(n, helper)
    u = u / np.linalg.norm(u)
    v = np.cross(n, u)
    grid = np.linspace(-span, span, resolution)
    U, V = np.meshgrid(grid, grid)
    pts = origin[None, None, :] + U[:, :, None] * u + V[:, :, None] * v
    return pts[:, :, 0], pts[:, :, 1], pts[:, :, 2]


def make_camera_pair_for_uncertainty(baseline=2.2, target=np.array([0.0, 0.0, 4.0])):
    K = make_calibration(850.0, 830.0, 320.0, 240.0)
    C1 = np.array([-baseline / 2.0, 0.15, -4.0])
    C2 = np.array([baseline / 2.0, 0.15, -4.0])
    P1 = camera_matrix(K, look_at_rotation(C1, target), C1)
    P2 = camera_matrix(K, look_at_rotation(C2, target), C2)
    return P1, P2

## 1. Noisy Correspondences Make Skew Rays

With exact image points, the two back-projected rays meet at the 3D point. With measured points, the rays normally become skew. The closest segment between the two rays is a Euclidean diagnostic: it tells us how large the inconsistency is, but it is not itself a projective-invariant objective. The 3D view below shows a single noisy match, its two rays, the midpoint estimate, the homogeneous DLT estimate, and the optimal image-corrected estimate.

In [ ]:
K, P1, P2, points3d, x1_clean, x2_clean, x1_noisy, x2_noisy = make_structure_scene()
F = fundamental_from_cameras(P1, P2)

X_dlt = linear_triangulate(P1, P2, x1_noisy, x2_noisy)
X_inhom = inhomogeneous_triangulate(P1, P2, x1_noisy, x2_noisy)
X_mid, ray_gaps = midpoint_triangulate(P1, P2, x1_noisy, x2_noisy)
x1_sampson, x2_sampson = sampson_correct(F, x1_noisy, x2_noisy)
x1_opt, x2_opt = optimal_correct(F, x1_noisy, x2_noisy)
X_opt = linear_triangulate(P1, P2, x1_opt, x2_opt)
X_refined = refine_structure_points(P1, P2, X_dlt, x1_noisy, x2_noisy)

example = int(np.argmax(ray_gaps))
C1, d1 = ray_from_image(P1, x1_noisy[example])
C2, d2 = ray_from_image(P2, x2_noisy[example])
q1, q2, ray_midpoint, ray_gap = closest_points_on_rays(C1, d1, C2, d2)
segment_len = 8.0

fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=points3d[:, 0], y=points3d[:, 1], z=points3d[:, 2],
    mode="markers", name="true structure", marker=dict(size=3, color="#6b7280")
))
fig.add_trace(go.Scatter3d(
    x=[C1[0], C2[0]], y=[C1[1], C2[1]], z=[C1[2], C2[2]],
    mode="markers+text", name="camera centers", text=["C1", "C2"], textposition="top center",
    marker=dict(size=6, color="#111827")
))
add_plotly_segment(fig, C1, C1 + segment_len * d1, "ray from noisy x", "#2563eb", 6)
add_plotly_segment(fig, C2, C2 + segment_len * d2, "ray from noisy x'", "#dc2626", 6)
add_plotly_segment(fig, q1, q2, "closest segment", "#f59e0b", 7, "dash")
method_points = np.vstack([points3d[example], ray_midpoint, X_dlt[example], X_opt[example], X_refined[example]])
fig.add_trace(go.Scatter3d(
    x=method_points[:, 0], y=method_points[:, 1], z=method_points[:, 2],
    mode="markers+text", name="candidate points",
    text=["truth", "midpoint", "DLT", "optimal", "refined"], textposition="top center",
    marker=dict(size=[6, 6, 6, 7, 7], color=["#16a34a", "#f59e0b", "#7c3aed", "#0891b2", "#be123c"]),
))
fig.update_layout(
    title="Skew rays and competing structure estimates for one noisy match",
    scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z", aspectmode="data"),
    legend=dict(orientation="h", y=0.02),
    margin=dict(l=0, r=0, t=40, b=0),
)
rays_path = save_plotly_html(fig, TOPIC, "interactive", "skew-rays-triangulation-methods.html")
artifact_paths.append(rays_path)
display_artifact(rays_path, width=900, height=620)

ray_gap

## 2. Linear Triangulation Is Useful, But Its Objective Is Algebraic

The four DLT equations are the chapter's simplest triangulation device. They are excellent initializers, especially when many views will later be used, but their residual is algebraic. The midpoint method uses a Euclidean distance in object space; that is meaningful for calibrated Euclidean reconstructions but has no projective meaning. The table below keeps the distinction visible by comparing image reprojection error, world error against the synthetic truth, and cheirality.

In [ ]:
method_rows = []
method_points_by_name = {
    "homogeneous_dlt": X_dlt,
    "inhomogeneous_w_equals_1": X_inhom,
    "ray_midpoint": X_mid,
    "optimal_corrected_dlt": X_opt,
    "nonlinear_refinement": X_refined,
}
for name, X in method_points_by_name.items():
    d1 = camera_depths(P1, X)
    d2 = camera_depths(P2, X)
    method_rows.append({
        "method": name,
        "reprojection_rmse_pixels": reprojection_rmse(P1, P2, X, x1_noisy, x2_noisy),
        "mean_world_error": mean_world_error(X, points3d),
        "positive_depth_fraction": float(np.mean((d1 > 0.0) & (d2 > 0.0))),
        "min_depth_camera_1": float(np.min(d1)),
        "min_depth_camera_2": float(np.min(d2)),
    })

method_table_path = save_csv(method_rows, TOPIC, "tables", "triangulation-method-comparison.csv")
artifact_paths.append(method_table_path)
display_artifact(method_table_path)
method_rows

### Projective-Frame Sanity Check

A triangulation rule is projective-invariant if transforming the world frame and transforming the cameras give the corresponding transformed point. Image-space geometric correction has this property because it changes only the measured image points before triangulating. Linear DLT does not minimize a projectively invariant norm. The short diagnostic below applies a non-affine 3D projective transform to the cameras and compares the triangulated result with the transformed original estimate.

In [ ]:
H = np.array([
    [1.15, 0.08, -0.03, 0.20],
    [0.02, 0.95, 0.05, -0.10],
    [0.04, -0.02, 1.10, 0.15],
    [0.015, -0.010, 0.018, 1.00],
])
Hinv = np.linalg.inv(H)
P1_H = P1 @ Hinv
P2_H = P2 @ Hinv
X_dlt_H = linear_triangulate(P1_H, P2_H, x1_noisy[[example]], x2_noisy[[example]])[0]
X_inhom_H = inhomogeneous_triangulate(P1_H, P2_H, x1_noisy[[example]], x2_noisy[[example]])[0]
X_opt_H = linear_triangulate(P1_H, P2_H, x1_opt[[example]], x2_opt[[example]])[0]


def transform_point(H, X):
    return dehomogenize((H @ homogenize(X.reshape(1, 3))[0]).reshape(1, 4))[0]


projective_invariance = {
    "homogeneous_dlt_frame_gap": float(np.linalg.norm(X_dlt_H - transform_point(H, X_dlt[example]))),
    "inhomogeneous_frame_gap": float(np.linalg.norm(X_inhom_H - transform_point(H, X_inhom[example]))),
    "optimal_corrected_dlt_frame_gap": float(np.linalg.norm(X_opt_H - transform_point(H, X_opt[example]))),
}
projective_invariance

## 3. Image-Space Correction: Raw, Sampson, And Optimal

The geometric cost in this chapter lives in the images. A corrected pair should lie on corresponding epipolar lines and stay near the measured points. The Sampson step is the first-order correction to the epipolar variety. OpenCV's `correctMatches` performs the standard optimal two-view correction: after it, the corrected points satisfy the epipolar equation to machine precision. Inspect the arrows: each correction is small in pixel units, but it changes the back-projected rays from skew to intersecting.

In [ ]:
correction_index = example
line2_raw = F @ homogenize(x1_noisy[[correction_index]])[0]
line2_opt = F @ homogenize(x1_opt[[correction_index]])[0]
line1_raw = F.T @ homogenize(x2_noisy[[correction_index]])[0]
line1_opt = F.T @ homogenize(x2_opt[[correction_index]])[0]

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
for ax, image_points, raw, sampson, optimal, raw_line, opt_line, title in [
    (axes[0], x1_noisy, x1_noisy[correction_index], x1_sampson[correction_index], x1_opt[correction_index], line1_raw, line1_opt, "image 1"),
    (axes[1], x2_noisy, x2_noisy[correction_index], x2_sampson[correction_index], x2_opt[correction_index], line2_raw, line2_opt, "image 2"),
]:
    pad = 35
    xlim = (image_points[:, 0].min() - pad, image_points[:, 0].max() + pad)
    ylim = (image_points[:, 1].min() - pad, image_points[:, 1].max() + pad)
    ax.scatter(image_points[:, 0], image_points[:, 1], s=14, color="#cbd5e1", label="other noisy matches")
    ax.scatter([raw[0]], [raw[1]], s=60, color="#111827", label="measured")
    ax.scatter([sampson[0]], [sampson[1]], s=55, color="#7c3aed", label="Sampson")
    ax.scatter([optimal[0]], [optimal[1]], s=55, color="#0891b2", label="optimal")
    ax.arrow(raw[0], raw[1], sampson[0] - raw[0], sampson[1] - raw[1], color="#7c3aed", width=0.05, head_width=2.5, length_includes_head=True)
    ax.arrow(raw[0], raw[1], optimal[0] - raw[0], optimal[1] - raw[1], color="#0891b2", width=0.05, head_width=2.5, length_includes_head=True)
    for line, color, label in [(raw_line, "#f97316", "raw epiline"), (opt_line, "#0891b2", "corrected epiline")]:
        seg = epiline_segment(line, xlim, ylim)
        if seg is not None:
            ax.plot(seg[:, 0], seg[:, 1], color=color, lw=1.8, label=label)
    ax.set_title(title)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.invert_yaxis()
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("u pixels")
    ax.set_ylabel("v pixels")
axes[1].legend(loc="upper right", fontsize=8)
correction_path = save_matplotlib(fig, TOPIC, "figures", "sampson-optimal-epipolar-corrections.png")
plt.close(fig)
artifact_paths.append(correction_path)
display_artifact(correction_path, width=900)

correction_metrics = {
    "raw_max_abs_epipolar_residual": float(np.max(np.abs(epipolar_residuals(F, x1_noisy, x2_noisy)))),
    "sampson_max_abs_epipolar_residual": float(np.max(np.abs(epipolar_residuals(F, x1_sampson, x2_sampson)))),
    "optimal_max_abs_epipolar_residual": float(np.max(np.abs(epipolar_residuals(F, x1_opt, x2_opt)))),
    "mean_sampson_step_pixels": float(np.mean(np.linalg.norm(x1_sampson - x1_noisy, axis=1) + np.linalg.norm(x2_sampson - x2_noisy, axis=1))),
    "mean_optimal_step_pixels": float(np.mean(np.linalg.norm(x1_opt - x1_noisy, axis=1) + np.linalg.norm(x2_opt - x2_noisy, axis=1))),
}
correction_metrics

## 4. The One-Parameter Geometric Error Can Have Local Minima

The chapter reduces the exact two-view correction to a one-parameter search over corresponding epipolar lines. The scalar cost below is the squared distance from the two measured points to a matched pair of epipolar lines after the usual translations and rotations. Plotting against `theta = arctan(t)` makes the whole infinite range of `t` visible. The roots of the derivative polynomial mark stationary points; the global minimum must be selected by evaluating the cost, not by trusting the first local descent.

In [ ]:
cost_params = dict(f=1.10, fp=0.90, a=2.0, b=3.0, c=3.0, d=4.0)
theta = np.linspace(-0.5 * np.pi + 1e-4, 0.5 * np.pi - 1e-4, 2400)
cost_values = geometric_cost(theta, **cost_params)
roots, derivative_poly = polynomial_roots_for_cost(**cost_params)
real_theta = []
for root in roots:
    if abs(root.imag) < 1e-8:
        real_theta.append(math.atan(float(root.real)))
real_theta = np.asarray(sorted(real_theta))
real_costs = geometric_cost(real_theta, **cost_params) if len(real_theta) else np.array([])
asymptotic_cost = (1.0 / cost_params["f"]**2) + (cost_params["c"]**2 / (cost_params["a"]**2 + cost_params["fp"]**2 * cost_params["c"]**2))

fig, ax = plt.subplots(figsize=(10, 4.8), constrained_layout=True)
ax.plot(theta, cost_values, color="#1f2937", lw=2)
if len(real_theta):
    ax.scatter(real_theta, real_costs, s=55, color="#dc2626", zorder=3, label="real derivative roots")
    best = int(np.argmin(real_costs))
    ax.scatter([real_theta[best]], [real_costs[best]], s=120, facecolor="none", edgecolor="#16a34a", lw=2.5, label="global checked minimum")
ax.set_xlabel("theta = arctan(t)")
ax.set_ylabel("squared image distance")
ax.set_title("One-parameter geometric triangulation cost")
ax.grid(True, alpha=0.25)
ax.legend(loc="upper right")
cost_path = save_matplotlib(fig, TOPIC, "figures", "one-parameter-geometric-error-cost.png")
plt.close(fig)
artifact_paths.append(cost_path)
display_artifact(cost_path, width=880)

cost_diagnostics = {
    "derivative_degree": int(derivative_poly.order),
    "real_stationary_points": int(len(real_theta)),
    "minimum_theta": float(real_theta[int(np.argmin(real_costs))]),
    "minimum_cost": float(np.min(real_costs)),
    "asymptotic_cost": float(asymptotic_cost),
    "minimum_beats_asymptote": bool(np.min(real_costs) <= asymptotic_cost),
}
cost_diagnostics

## 5. Cheirality, Ray Angle, And Reconstruction Uncertainty

A triangulated point is not acceptable just because it has a small residual. It should also lie in front of the cameras. Among valid points, the largest uncertainty direction is usually close to the viewing rays. The visualization uses the inverse normal matrix `(J.T @ J)^-1` from the two projection Jacobians. The narrow-angle point has a much longer uncertainty ellipsoid, which is the computational version of the chapter's rule of thumb: the angle between rays is a better warning than baseline alone.

In [ ]:
P1_u, P2_u = make_camera_pair_for_uncertainty(baseline=2.2)
wide_point = np.array([-0.35, 0.05, 2.2])
narrow_point = np.array([0.35, 0.05, 8.2])
uncertainty_points = np.vstack([wide_point, narrow_point])

fig = go.Figure()
C1u = camera_center_euclidean(P1_u)
C2u = camera_center_euclidean(P2_u)
fig.add_trace(go.Scatter3d(
    x=[C1u[0], C2u[0]], y=[C1u[1], C2u[1]], z=[C1u[2], C2u[2]],
    mode="markers+text", text=["C1", "C2"], textposition="top center", name="cameras",
    marker=dict(size=6, color="#111827"),
))
uncertainty_rows = []
colors = ["#16a34a", "#dc2626"]
labels = ["wide ray angle", "narrow ray angle"]
for X, color, label in zip(uncertainty_points, colors, labels):
    add_plotly_segment(fig, C1u, X, label + " ray 1", color, 4)
    add_plotly_segment(fig, C2u, X, label + " ray 2", color, 4, "dash")
    cov = point_covariance(P1_u, P2_u, X, sigma_pixels=1.0)
    eigvals, eigvecs = np.linalg.eigh(cov)
    eigvals = np.maximum(eigvals, 0.0)
    angle = ray_angle_degrees(P1_u, P2_u, X)
    uncertainty_rows.append({
        "label": label,
        "ray_angle_degrees": angle,
        "std_axes_world_units": [float(v) for v in np.sqrt(eigvals)],
        "largest_to_smallest_std_ratio": float(np.sqrt(eigvals[-1]) / max(np.sqrt(eigvals[0]), 1e-12)),
    })
    u = np.linspace(0.0, 2.0 * np.pi, 28)
    v = np.linspace(0.0, np.pi, 14)
    sphere = np.stack([
        np.outer(np.cos(u), np.sin(v)),
        np.outer(np.sin(u), np.sin(v)),
        np.outer(np.ones_like(u), np.cos(v)),
    ], axis=-1)
    scale = 45.0
    ellipsoid = X + scale * (sphere @ (eigvecs @ np.diag(np.sqrt(eigvals))).T)
    fig.add_trace(go.Surface(
        x=ellipsoid[:, :, 0], y=ellipsoid[:, :, 1], z=ellipsoid[:, :, 2],
        showscale=False, opacity=0.35, name=label + " covariance", surfacecolor=np.zeros(ellipsoid.shape[:2]),
        colorscale=[[0, color], [1, color]],
    ))
    fig.add_trace(go.Scatter3d(
        x=[X[0]], y=[X[1]], z=[X[2]], mode="markers+text", text=[label], textposition="top center",
        marker=dict(size=5, color=color), name=label + " point",
    ))
fig.update_layout(
    title="Ray angle controls the shape of 3D point uncertainty",
    scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z", aspectmode="data"),
    margin=dict(l=0, r=0, t=40, b=0),
)
uncertainty_path = save_plotly_html(fig, TOPIC, "interactive", "ray-angle-uncertainty-ellipsoids.html")
artifact_paths.append(uncertainty_path)
display_artifact(uncertainty_path, width=900, height=620)

uncertainty_rows

### Ray-Angle Scan

The ellipsoids show two examples. The scan below samples a range of depths and lateral offsets, then compares ray angle to the largest covariance standard deviation. The monotone trend is not a theorem for every camera arrangement, but it is the diagnostic pattern this chapter wants the reader to check before trusting distant structure.

In [ ]:
scan_rows = []
for z in np.linspace(2.0, 10.0, 24):
    for x in np.linspace(-0.8, 0.8, 9):
        X = np.array([x, 0.05, z])
        cov = point_covariance(P1_u, P2_u, X, sigma_pixels=1.0)
        eigvals = np.linalg.eigvalsh(cov)
        scan_rows.append({
            "x": float(x),
            "z": float(z),
            "ray_angle_degrees": ray_angle_degrees(P1_u, P2_u, X),
            "largest_std_world_units": float(np.sqrt(max(eigvals[-1], 0.0))),
        })
angles = np.array([row["ray_angle_degrees"] for row in scan_rows])
largest_std = np.array([row["largest_std_world_units"] for row in scan_rows])

fig, ax = plt.subplots(figsize=(7.6, 4.8), constrained_layout=True)
scatter = ax.scatter(angles, largest_std, c=[row["z"] for row in scan_rows], cmap="viridis", s=38)
ax.set_yscale("log")
ax.set_xlabel("ray angle in degrees")
ax.set_ylabel("largest covariance std, world units")
ax.set_title("Smaller ray angle gives longer depth uncertainty")
ax.grid(True, which="both", alpha=0.25)
cb = fig.colorbar(scatter, ax=ax)
cb.set_label("point depth coordinate Z")
scan_path = save_matplotlib(fig, TOPIC, "figures", "ray-angle-depth-uncertainty-scan.png")
plt.close(fig)
artifact_paths.append(scan_path)
display_artifact(scan_path, width=760)

ray_angle_uncertainty_correlation = float(np.corrcoef(angles, np.log(largest_std))[0, 1])
ray_angle_uncertainty_correlation

## 6. Applied Lab: Structure Refinement From A Linear Start

This lab is the chapter's recommended workflow in miniature. Use linear triangulation to get a finite starting point, then refine the 3D point by minimizing geometric reprojection error. The optimal image-corrected triangulation and the nonlinear refinement agree closely in this two-view setup because both target the same image-space objective. The bar chart shows that the improvement is small but systematic for this controlled noise level.

In [ ]:
rmse_before = []
rmse_after = []
for X0, Xr, a, b in zip(X_dlt, X_refined, x1_noisy, x2_noisy):
    rmse_before.append(np.linalg.norm(projection_residual_vector(P1, P2, X0, a, b)))
    rmse_after.append(np.linalg.norm(projection_residual_vector(P1, P2, Xr, a, b)))
rmse_before = np.asarray(rmse_before)
rmse_after = np.asarray(rmse_after)
order = np.argsort(rmse_before - rmse_after)[::-1]

fig, ax = plt.subplots(figsize=(10, 4.8), constrained_layout=True)
idx = np.arange(len(order))
ax.bar(idx - 0.18, rmse_before[order], width=0.36, color="#94a3b8", label="linear DLT")
ax.bar(idx + 0.18, rmse_after[order], width=0.36, color="#be123c", label="refined")
ax.set_xlabel("point index, sorted by improvement")
ax.set_ylabel("two-view residual norm, pixels")
ax.set_title("Structure refinement lowers geometric reprojection residual")
ax.legend()
refinement_path = save_matplotlib(fig, TOPIC, "figures", "structure-refinement-reprojection-drop.png")
plt.close(fig)
artifact_paths.append(refinement_path)
display_artifact(refinement_path, width=860)

refinement_metrics = {
    "linear_total_rmse_pixels": reprojection_rmse(P1, P2, X_dlt, x1_noisy, x2_noisy),
    "refined_total_rmse_pixels": reprojection_rmse(P1, P2, X_refined, x1_noisy, x2_noisy),
    "mean_pointwise_residual_drop_pixels": float(np.mean(rmse_before - rmse_after)),
    "all_refined_no_worse": bool(np.all(rmse_after <= rmse_before + 1e-8)),
}
refinement_metrics

## 7. Line Reconstruction From Two Image Lines

A 3D line has four degrees of freedom. Each image line supplies a back-projected plane, so two views determine the line by plane intersection. The degeneracy is different from point triangulation: if the 3D line lies in an epipolar plane, both image lines pass through their epipoles and the two planes collapse toward the same plane. The visualization shows the nondegenerate case, while the numeric check below also measures the epipolar-plane warning case.

In [ ]:
line_A = np.array([-0.75, -0.25, 2.9])
line_B = np.array([0.85, 0.55, 4.2])
l1 = line_from_projected_endpoints(P1, line_A, line_B)
l2 = line_from_projected_endpoints(P2, line_A, line_B)
plane1 = P1.T @ l1
plane2 = P2.T @ l2
plane_stack = np.vstack([plane1, plane2])
U, V = nullspace_line_points(plane_stack)
reconstructed_line_pts = affine_points_from_line_basis(U, V, count=2)

fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=[C1[0], C2[0]], y=[C1[1], C2[1]], z=[C1[2], C2[2]],
    mode="markers+text", text=["C1", "C2"], textposition="top center", name="cameras",
    marker=dict(size=6, color="#111827"),
))
for plane, color, name in [(plane1, "#2563eb", "back-projected plane from image 1"), (plane2, "#dc2626", "back-projected plane from image 2")]:
    Xs, Ys, Zs = plane_surface(plane, span=3.0, resolution=10)
    fig.add_trace(go.Surface(x=Xs, y=Ys, z=Zs, showscale=False, opacity=0.22, name=name, colorscale=[[0, color], [1, color]]))
add_plotly_segment(fig, line_A, line_B, "true 3D line", "#16a34a", 7)
if len(reconstructed_line_pts) == 2:
    add_plotly_segment(fig, reconstructed_line_pts[0], reconstructed_line_pts[1], "SVD plane intersection", "#f59e0b", 5, "dash")
fig.update_layout(
    title="A 3D line as the intersection of two back-projected image planes",
    scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z", aspectmode="data"),
    margin=dict(l=0, r=0, t=40, b=0),
)
line_path = save_plotly_html(fig, TOPIC, "interactive", "line-reconstruction-backprojected-planes.html")
artifact_paths.append(line_path)
display_artifact(line_path, width=900, height=620)

# Degenerate warning: a line through the baseline lies in an epipolar plane.
baseline_point = 0.45 * C1 + 0.55 * C2
off_baseline_point = baseline_point + np.array([0.30, 1.20, 1.40])
dl1 = line_from_projected_endpoints(P1, baseline_point, off_baseline_point)
dl2 = line_from_projected_endpoints(P2, baseline_point, off_baseline_point)
dplane_stack = np.vstack([P1.T @ dl1, P2.T @ dl2])
line_singular_values = linalg.svd(plane_stack, compute_uv=False)
degenerate_singular_values = linalg.svd(dplane_stack, compute_uv=False)
line_metrics = {
    "nondegenerate_plane_stack_singular_values": [float(v) for v in line_singular_values],
    "epipolar_plane_stack_singular_values": [float(v) for v in degenerate_singular_values],
    "true_line_plane_incidence_max": float(np.max(np.abs(plane_stack @ homogenize(np.vstack([line_A, line_B])).T))),
    "nondegenerate_rank_warning_ratio": float(line_singular_values[1] / line_singular_values[0]),
    "epipolar_rank_warning_ratio": float(degenerate_singular_values[1] / max(degenerate_singular_values[0], 1e-12)),
}
line_metrics

## Final Sanity Checks

The final cell keeps the chapter honest: artifacts must exist and have nonzero size, optimal correction must enforce the epipolar constraint, cheirality must hold for the synthetic scene, refinement must not increase the pointwise residual, covariance matrices must be positive semidefinite, and the line reconstruction must satisfy plane incidence.

In [ ]:
cov_eigenvalues = []
for X in uncertainty_points:
    cov_eigenvalues.extend(np.linalg.eigvalsh(point_covariance(P1_u, P2_u, X, sigma_pixels=1.0)).tolist())

summary = {
    "source_span": "printed pages 310-324; PDF pages 328-342",
    "artifact_count": len(artifact_paths),
    "artifacts": [relative_to_book(path, BOOK_ROOT) for path in artifact_paths],
    "method_rows": method_rows,
    "projective_invariance_diagnostic": projective_invariance,
    "correction_metrics": correction_metrics,
    "cost_diagnostics": cost_diagnostics,
    "uncertainty_rows": uncertainty_rows,
    "ray_angle_log_uncertainty_correlation": ray_angle_uncertainty_correlation,
    "refinement_metrics": refinement_metrics,
    "line_metrics": line_metrics,
}
final_sanity = summary
summary_path = save_json(summary, TOPIC, "checks", "structure-computation-sanity-summary.json")
check_paths.append(summary_path)

assert_artifacts(artifact_paths + check_paths, min_bytes=64)
assert correction_metrics["optimal_max_abs_epipolar_residual"] < 1e-9
assert correction_metrics["sampson_max_abs_epipolar_residual"] < correction_metrics["raw_max_abs_epipolar_residual"]
assert all(row["positive_depth_fraction"] == 1.0 for row in method_rows)
assert refinement_metrics["refined_total_rmse_pixels"] <= refinement_metrics["linear_total_rmse_pixels"] + 1e-9
assert refinement_metrics["all_refined_no_worse"]
assert min(cov_eigenvalues) > -1e-12
assert cost_diagnostics["derivative_degree"] == 6
assert cost_diagnostics["real_stationary_points"] >= 3
assert cost_diagnostics["minimum_beats_asymptote"]
assert line_metrics["true_line_plane_incidence_max"] < 1e-7
assert line_metrics["epipolar_rank_warning_ratio"] < line_metrics["nondegenerate_rank_warning_ratio"]

summary_path

## Takeaways

- Noisy two-view matches usually define skew rays, so triangulation is an estimation problem.
- Homogeneous DLT is a strong initializer, but its algebraic objective is not the same as geometric image error.
- Sampson correction is a fast first-order image correction; optimal correction enforces the epipolar constraint exactly within numerical precision.
- Cheirality and ray angle are essential diagnostics: a low residual with poor depth geometry is still a fragile reconstruction.
- A 3D line is naturally recovered as the intersection of two back-projected image planes, except when those planes become the same epipolar plane.